### Tool Caling

In [1]:
# tool create
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv

load_dotenv()

True

In [3]:

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  Fetch the LIVE exchange rate from the ExchangeRate API.
  Use this tool ONLY when there is no exchange rate provide.
  """
  
  url = f'https://v6.exchangerate-api.com/v6/4edb9626e017d813681eeba8/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def calculate(
    base_currency_value: float,
    conversion_rate: float
) -> float:
    """
    Calculate the converted currency amount using a conversion rate
    already provided by the user. Do not fetch exchange rates.
    """
    return base_currency_value * conversion_rate

In [ ]:
get_conversion_factor.invoke({'base_currency':'INR','target_currency':'USD'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1795305601,
 'time_last_update_utc': 'Sun, 22 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1795392001,
 'time_next_update_utc': 'Mon, 23 Nov 2026 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'EUR',
 'conversion_rate': 0.009812}

In [5]:
calculate.invoke({'base_currency_value':10, 'conversion_rate':93.5163})

935.163

In [6]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [7]:
llm_with_tools = llm.bind_tools([get_conversion_factor, calculate])

In [8]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that convert 10 inr to usd. If you want you can call the all tools at once if needed to get the result caz it has no agent loop')]

In [9]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that convert 10 inr to usd. If you want you can call the all tools at once if needed to get the result caz it has no agent loop', additional_kwargs={}, response_metadata={})]

In [10]:
ai_message = llm_with_tools.invoke(messages)

In [11]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_142f9dc0-34df-46bc-971c-9700ed7e95cb',
  'type': 'tool_call'}]

In [12]:
messages1 = [
    HumanMessage(
        content="What is the conversion factor between USD and INR, and based on that convert 10 INR to USD"
    ),

    AIMessage(
        content="",
        additional_kwargs={
            "reasoning_content": "We need to fetch the live exchange rate INR to USD. Then convert 10 INR to USD.",
            "tool_calls": [
                {
                    "id": "fc_a74debf7-3070-49d9-a02c-9704bb81884a",
                    "function": {
                        "arguments": '{"base_currency":"INR","target_currency":"USD"}',
                        "name": "get_conversion_factor"
                    },
                    "type": "function"
                }
            ]
        },
        response_metadata={
            "finish_reason": "tool_calls",
            "model_provider": "groq"
        },
        id="lc_run--01a1f551-23b9-73d4-ac98-2e8074c9117d-0",
        tool_calls=[
            {
                "name": "get_conversion_factor",
                "args": {
                    "base_currency": "INR",
                    "target_currency": "USD"
                },
                "id": "fc_a74debf7-3070-49d9-a02c-9704bb81884a",
                "type": "tool_call"
            }
        ]
    ),

    ToolMessage(
        content=str({
            "result": "success",
            "base_code": "INR",
            "target_code": "USD",
            "conversion_rate": 0.01069
        }),
        tool_call_id="fc_a74debf7-3070-49d9-a02c-9704bb81884a"
    )
]

In [13]:
ai_message1 = llm_with_tools.invoke(messages1)

In [14]:
ai_message1.tool_calls

[{'name': 'calculate',
  'args': {'base_currency_value': 10, 'conversion_rate': 0.01069},
  'id': 'fc_834a1643-fcf8-4cc1-9d71-6b0758e1aa1f',
  'type': 'tool_call'}]

In [16]:
tools = [get_conversion_factor, calculate]

llm_with_tools = llm.bind_tools(tools)

tools_by_name = {
    tool.name: tool
    for tool in tools
}

In [30]:

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage


def run_agent(user_query: str, max_iterations: int = 5):

    messages = [
        SystemMessage(
            content="""
            You are a helpful currency conversion agent.

            Rules:
            1. If the user hasn't provided an exchange rate, use get_conversion_factor to fetch the live exchange rate.
            2. If the user requests a currency conversion, use calculate to calculate the converted amount using the exchange rate.
            3. Never invent or assume an exchange rate.
            4. If the user provides an exchange rate, use that rate directly with calculate instead of fetching a new one.
            5. Give the final answer only after completing all required tool calls.
            6. Respond in a natural paragraph. Do not use unnecessary headings, bullet points, or structured formatting.
            7. If the user misspells a currency name, correct it to the appropriate currency name or ISO 4217 currency code before calling a tool. For example, "Euros" or "Euro" should be normalized to EUR, "US dollars" to USD, and "Indian rupees" to INR.
            8. If the user's currency is ambiguous, ask them for clarification rather than guessing.
            9. If the user only asks for an exchange rate and not a conversion of an amount, return the exchange rate without calling calculate.
            10. Keep the final response concise, accurate, and easy to understand.
            """
        ),
        HumanMessage(content=user_query)
    ]

    for iteration in range(max_iterations):

        # 1. Ask the LLM what to do
        ai_message = llm_with_tools.invoke(messages)

        # 2. Add the LLM response to conversation history
        messages.append(ai_message)

        # 3. Check whether the LLM wants to call tools
        if not ai_message.tool_calls:

            return ai_message.content

        # 4. Execute requested tools
        for tool_call in ai_message.tool_calls:
            print(f"Tool call: {tool_call["name"]}")
            selected_tool = tools_by_name.get(tool_call["name"])

            if selected_tool is None:
                result = f"Unknown tool: {tool_call['name']}"
            else:
                try:
                    result = selected_tool.invoke(
                        tool_call["args"]
                    )
                except Exception as e:
                    result = f"Tool execution error: {e}"

            # 5. Send tool result back to the LLM
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Maximum iterations reached. Unable to complete the task."


In [32]:
# Run the agent
response = run_agent(
    "What is the conversion factor between inr and euro? "
    "Based on that, convert 10 inr to euro."
)

print(response)

Tool call: get_conversion_factor
Tool call: calculate
The current exchange rate from Indian rupees (INR) to euros (EUR) is about 0.009812. Using that rate, 10 INR converts to roughly 0.09812 EUR.
